In [22]:
from parser import Parser
import shapely
from shapely.ops import transform
import osmnx as ox
import geopandas as gpd

In [ ]:
custom_filter = (
    f'["highway"]["area"!~"yes"]{ox.settings.default_access}'
    f'["highway"!~"abandoned|bridleway|bus_guideway|construction|corridor|cycleway|elevator|'
    f"escalator|footway|no|path|pedestrian|planned|platform|proposed|raceway|razed|"
    f'steps|track"]'
    f'["motor_vehicle"!~"no"]["motorcar"!~"no"]'
    f'["service"!~"alley|driveway|emergency_access|parking|parking_aisle|private"]'
)


graph = ox.graph_from_place("Санкт-Петербург", custom_filter=custom_filter, simplify=False)  # TODO: union with kad
nodes, edges = ox.graph_to_gdfs(graph)

In [3]:
bus_parser = Parser.BusGraphParser("Санкт-Петербург")
example_route = bus_parser.get_route("/spb/bus/61")

read https://kudikina.ru/spb/bus/61/map from cache


In [4]:
example_point = example_route[0]
example_point = shapely.geometry.Point(example_point[1], example_point[0])
with open("example_point.geojson", "w") as f:
    f.write(shapely.to_geojson(example_point))

In [ ]:
def find_near_points2(point, gdf, tolerance=0.01):
    # Create a buffer around the point
    buffer = point.buffer(tolerance)
    # Convert the buffer to a GeoDataFrame
    buffer_gdf = gpd.GeoDataFrame(geometry=[buffer], crs=gdf.crs)
    # Use the buffer to find points within the tolerance
    within_points = gpd.sjoin(gdf, buffer_gdf, predicate = 'within')


    return within_points

In [30]:
%%timeit


# Find points near to example_point
near_points = find_near_points(example_point, nodes, tolerance=0.0005)


12.2 ms ± 225 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [31]:
%%timeit


# Find points near to example_point
near_points2 = find_near_points2(example_point, nodes, tolerance=0.0005)


1.33 ms ± 4.91 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
near_points.to_file("points.geojson", driver="GeoJSON")

In [15]:
# all_routes = {}
# for route_info in bus_parser.get_all_routes_info():

#     route_url = route_info[2]
#     all_routes[route_url] = bus_parser.get_route(route_url)

In [16]:
def flip(x, y):
    return y, x

In [17]:
l = shapely.LineString(example_route)
l = transform(flip, l)

In [18]:
with open("example_route.geojson", "w") as f:
    f.write(shapely.to_geojson(l))

In [19]:
edges.to_file("edges.geojson", driver="GeoJSON")
nodes.to_file("nodes.geojson", driver="GeoJSON")